# Sen1Floods11, Sentinel-1 only: computer-vision tricks against false positives (leave-one-event-out)

No Earth Engine needed. One U-Net (resnet34), same 11-event leave-one-event-out (LOEO) protocol as before, one change at a time against the same base recipe:

| recipe | change | why |
|---|---|---|
| `base` | BCE + Dice, 256 px crops, +-1 dB shift | reference (same as earlier runs) |
| `aug+` | random contrast (gain 0.85-1.15 around the patch mean) and +-2 dB shift | the original Sen1Floods11 paper reports contrast/brightness augmentation helping; events differ in backscatter offsets |
| `tversky` | BCE + Tversky (FP weight 0.7, FN weight 0.3) | penalise false positives harder |
| `ctx512` | train on full 512 px chips (batch 8) | more context to tell wet soil / shadow from water |
| `ctx512+aux` | + chip-level "contains water" head; at test the chip's mask is zeroed if that head says no water | the worst events fail on false positives; test it with and without the gate |

After training, **post-processing** is evaluated for every recipe: removing connected components smaller than k pixels (k fixed in advance: 0, 10, 25, 50, 100, 200), reported as a trend, not tuned on the held-out events.

Everything is a **hypothesis screen with one seed**: differences below about 0.015 macro IoU are not distinguishable from seed noise in your earlier runs. Treat a recipe as promising only if it helps on several events *and* the paired interval stays above 0, then confirm with more seeds.

In [ ]:
import os, sys, glob, re, json, time, random, shutil, subprocess, warnings, io, copy, math
warnings.filterwarnings("ignore")
T0 = time.time()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch", "rasterio", "earthengine-api"], check=False)

import numpy as np, pandas as pd, torch, torch.nn.functional as F
import rasterio
import segmentation_models_pytorch as smp

DEV = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = True
print("device:", DEV, "| torch", torch.__version__, "| smp", smp.__version__)

In [ ]:

CFG = dict(
    SMOKE=False,
    ENC="unet:resnet34", STEPS=1500, SEEDS=[0],
    RECIPES={
        "base":       dict(aug="basic",  loss="bce_dice",    aux=False, crop=256, bs=24),
        "aug+":       dict(aug="strong", loss="bce_dice",    aux=False, crop=256, bs=24),
        "tversky":    dict(aug="basic",  loss="bce_tversky", aux=False, crop=256, bs=24),
        "ctx512":     dict(aug="basic",  loss="bce_dice",    aux=False, crop=512, bs=8),
        "ctx512+aux": dict(aug="basic",  loss="bce_dice",    aux=True,  crop=512, bs=8),
    },
    POST_K=[0, 10, 25, 50, 100, 200],
    LR=3e-4, WD=1e-4, EMA=0.995, EVAL_EVERY=250, ENC_WEIGHTS="imagenet", SIZE=512, DEADLINE_H=10.5,
    OUT="/kaggle/working/out", ROOT="/kaggle/working/s1f11",
)
if CFG["SMOKE"]: CFG.update(STEPS=30, EVAL_EVERY=15)
os.makedirs(CFG["OUT"], exist_ok=True)
def hours(): return (time.time() - T0) / 3600
def set_seed(s): random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


In [ ]:
# ---------- hand-labelled chips + official splits ----------
ROOT, B = CFG["ROOT"], "gs://sen1floods11/v1.1"
def sh(c):
    print("$", c); return subprocess.run(c, shell=True, capture_output=True, text=True)

if len(glob.glob(f"{ROOT}/S1Hand/*.tif")) < 446:
    if shutil.which("gsutil") is None: sh("pip install -q gsutil")
    os.makedirs(ROOT, exist_ok=True)
    sh(f"gsutil -m -q cp -r {B}/splits/flood_handlabeled {ROOT}/")
    for d in ["S1Hand", "LabelHand"]:
        print(sh(f"gsutil -m -q cp -r {B}/data/flood_events/HandLabeled/{d} {ROOT}/").stderr[-300:])

def find_csv(name):
    p = glob.glob(f"{ROOT}/**/{name}", recursive=True); assert p, f"missing {name}"; return p[0]
def read_split(name):
    df = pd.read_csv(find_csv(name), header=None)
    return [(f"{ROOT}/S1Hand/{a}", f"{ROOT}/LabelHand/{b}") for a, b in zip(df[0], df[1])]

SPLIT = {k: read_split(f"flood_{k}_data.csv") for k in ["train", "valid", "test", "bolivia"]}
SPLIT["bol"] = SPLIT.pop("bolivia")
ITEMS, IDX, pos = [], {}, 0
for k in ["train", "valid", "test", "bol"]:
    IDX[k] = np.arange(pos, pos + len(SPLIT[k])); ITEMS += SPLIT[k]; pos += len(SPLIT[k])
cid = lambda p: re.sub(r"_(S1Hand|LabelHand)\.tif$", "", os.path.basename(p))
IDS = [cid(a) for a, _ in ITEMS]
EVENT = np.array([i.split("_")[0] for i in IDS])
EVENTS = sorted(set(EVENT))
print({k: len(v) for k, v in IDX.items()}, "| events:", EVENTS)
assert len(ITEMS) == len(set(IDS)) and all(os.path.exists(a) and os.path.exists(b) for a, b in ITEMS)
if CFG["SMOKE"]: print("SMOKE: all chips are still used for terrain fetch / LOEO folds are limited below")

In [ ]:

# ---------- S1-only input stack: [VV, VH, VV-VH] ----------
S = CFG["SIZE"]; N = len(ITEMS)
def read_tif(p):
    with rasterio.open(p) as s: return s.read().astype(np.float32)
XR = np.zeros((N, 3, S, S), np.float32); Y = np.zeros((N, S, S), np.int8)
for n, (a, b) in enumerate(ITEMS):
    s1 = read_tif(a)[:2]
    s1[0] = np.where(np.isfinite(s1[0]), s1[0], -10.0); s1[1] = np.where(np.isfinite(s1[1]), s1[1], -17.0)   # nodata -> typical land
    s1 = np.clip(s1, -35.0, 5.0)
    XR[n, 0], XR[n, 1], XR[n, 2] = s1[0], s1[1], s1[0] - s1[1]
    Y[n] = read_tif(b)[0].astype(np.int8)
MEAN = XR[IDX["train"]].mean((0, 2, 3), keepdims=True)[0]; STD = XR[IDX["train"]].std((0, 2, 3), keepdims=True)[0] + 1e-6
X = ((XR - MEAN[None]) / STD[None]).astype(np.float16); del XR
CH = [0, 1, 2]
print("X", X.shape, "| mean", MEAN.ravel().round(2), "std", STD.ravel().round(2), "| water share", float((Y == 1).sum() / (Y >= 0).sum()))


In [ ]:

# ---------- model / loss / metrics / training ----------
from scipy.ndimage import label as cc_label

def drop_small(b, k):
    lab, n = cc_label(b)
    if n == 0: return b
    sizes = np.bincount(lab.ravel()); keep = sizes >= k; keep[0] = False
    return keep[lab]

def make_model(spec, aux):
    arch, enc = spec.split(":")
    cls = {"unet": smp.Unet, "upp": smp.UnetPlusPlus}[arch]
    w = CFG["ENC_WEIGHTS"] if CFG["ENC_WEIGHTS"] != "none" else None
    kw = dict(aux_params=dict(classes=1, dropout=0.2)) if aux else {}
    return cls(enc, encoder_weights=w, in_channels=3, classes=1, **kw).to(DEV)

def seg_out(out): return out[0] if isinstance(out, (tuple, list)) else out
def aux_out(out): return out[1] if isinstance(out, (tuple, list)) else None

def masked_loss(out, y, kind, aux):
    logit = seg_out(out).float()
    m = (y >= 0).float(); t = (y == 1).float()
    bce = (F.binary_cross_entropy_with_logits(logit, t, reduction="none") * m).sum() / m.sum().clamp(min=1)
    p = torch.sigmoid(logit)
    if kind == "bce_tversky":
        tp = (p * t * m).sum(); fp = (p * (1 - t) * m).sum(); fn = ((1 - p) * t * m).sum()
        reg = 1 - (tp + 1) / (tp + 0.7 * fp + 0.3 * fn + 1)
    else:
        pm = p * m; inter = (pm * t).sum((1, 2, 3)); den = pm.sum((1, 2, 3)) + (t * m).sum((1, 2, 3))
        reg = 1 - ((2 * inter + 1) / (den + 1)).mean()
    loss = bce + reg
    if aux:
        has = ((t * m).sum((1, 2, 3)) / m.sum((1, 2, 3)).clamp(min=1) >= 0.005).float()
        loss = loss + 0.3 * F.binary_cross_entropy_with_logits(aux_out(out).float()[:, 0], has)
    return loss

def counts(prob, y, thr=0.5):
    m = y >= 0; pr = (prob >= thr) & m; t = (y == 1) & m
    return np.stack([(pr & t).sum((1, 2)), (pr & ~t).sum((1, 2)), (~pr & t).sum((1, 2))], 1)
def counts_bin(pr, y):
    m = y >= 0; pr = pr & m; t = (y == 1) & m
    return np.stack([(pr & t).sum((1, 2)), (pr & ~t).sum((1, 2)), (~pr & t).sum((1, 2))], 1)
def iou_c(c): return float(c[:, 0].sum() / max(c.sum(), 1)) if c.ndim == 2 else float(c[0] / max(c.sum(), 1))
def prec_c(c): return float(c[:, 0].sum() / max(c[:, 0].sum() + c[:, 1].sum(), 1))
def rec_c(c): return float(c[:, 0].sum() / max(c[:, 0].sum() + c[:, 2].sum(), 1))

class EMA:
    def __init__(s, model, d): s.m = copy.deepcopy(model).eval(); s.d = d; [p.requires_grad_(False) for p in s.m.parameters()]
    @torch.no_grad()
    def update(s, model, step):
        d = min(s.d, (1 + step) / (10 + step))
        for e, p in zip(s.m.state_dict().values(), model.state_dict().values()):
            if e.dtype.is_floating_point: e.mul_(d).add_(p.detach(), alpha=1 - d)
            else: e.copy_(p)

@torch.no_grad()
def predict(model, idx, aux, bs=8):
    model.eval(); P, A = [], []
    for i in range(0, len(idx), bs):
        x = torch.from_numpy(X[idx[i:i + bs]].astype(np.float32)).to(DEV)
        with torch.autocast("cuda", dtype=torch.float16, enabled=DEV == "cuda"):
            out = model(x)
        P.append(torch.sigmoid(seg_out(out).float())[:, 0].cpu().numpy().astype(np.float16))
        if aux: A.append(torch.sigmoid(aux_out(out).float())[:, 0].cpu().numpy())
    return np.concatenate(P), (np.concatenate(A) if aux else None)

def make_batch(idx, bs, crop, aug, rng):
    xs, ys = [], []
    for i in rng.choice(idx, bs):
        r, c = rng.integers(0, S - crop + 1), rng.integers(0, S - crop + 1)
        x = X[i][:, r:r + crop, c:c + crop].astype(np.float32); y = Y[i, r:r + crop, c:c + crop]
        if rng.random() < .5: x, y = x[:, :, ::-1], y[:, ::-1]
        if rng.random() < .5: x, y = x[:, ::-1], y[::-1]
        k = int(rng.integers(4))
        if k: x, y = np.rot90(x, k, (1, 2)), np.rot90(y, k)
        x = np.ascontiguousarray(x)
        if aug == "strong":                                    # contrast + shift in the dB domain, diff channel recomputed
            raw = x[:2] * STD[:2] + MEAN[:2]; mu = raw.mean((1, 2), keepdims=True)
            raw = np.clip(mu + rng.uniform(0.85, 1.15) * (raw - mu) + rng.uniform(-2, 2), -35.0, 5.0)
            x = np.stack([(raw[0] - MEAN[0, 0, 0]) / STD[0, 0, 0], (raw[1] - MEAN[1, 0, 0]) / STD[1, 0, 0],
                          ((raw[0] - raw[1]) - MEAN[2, 0, 0]) / STD[2, 0, 0]]).astype(np.float32)
        else:
            sh = rng.uniform(-1, 1); x[0] += sh / STD[0, 0, 0]; x[1] += sh / STD[1, 0, 0]
        xs.append(x); ys.append(np.ascontiguousarray(y))
    return torch.from_numpy(np.stack(xs)), torch.from_numpy(np.stack(ys).astype(np.float32))[:, None]

def train(rec, train_idx, steps, seed, tag=""):
    set_seed(seed); rng = np.random.default_rng(seed)
    model = make_model(CFG["ENC"], rec["aux"]); ema = EMA(model, CFG["EMA"])
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["LR"], weight_decay=CFG["WD"])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG["LR"], total_steps=steps, pct_start=0.1)
    scaler = torch.cuda.amp.GradScaler(enabled=DEV == "cuda"); t0, run = time.time(), 0.0
    for step in range(1, steps + 1):
        model.train()
        x, y = make_batch(train_idx, rec["bs"], rec["crop"], rec["aug"], rng)
        x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=DEV == "cuda"):
            out = model(x)
        loss = masked_loss(out, y, rec["loss"], rec["aux"])
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step(); ema.update(model, step)
        run += loss.item()
        if step % CFG["EVAL_EVERY"] == 0 or step == steps:
            print(f"[{tag}] step {step}/{steps} loss {run / min(CFG['EVAL_EVERY'], step):.4f} ({time.time() - t0:.0f}s)"); run = 0.0
    del model, opt; torch.cuda.empty_cache()
    return ema.m


In [ ]:
# ---------- stats helpers ----------
def boot_pooled(cA, cB=None, n=2000, seed=0):
    rng = np.random.default_rng(seed); k = len(cA); a = []; d = []
    for _ in range(n):
        i = rng.integers(0, k, k); ia = iou_c(cA[i]); a.append(ia)
        if cB is not None: d.append(ia - iou_c(cB[i]))
    ci = lambda v: (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5)))
    return (ci(a), (float(np.mean(d)), ci(d))) if cB is not None else ci(a)

def macro_iou(c, ev, ev_list):
    return float(np.mean([iou_c(c[ev == e]) for e in ev_list]))

def boot_macro(cA, cB, ev, ev_list, n=2000, seed=0):
    rng = np.random.default_rng(seed); groups = {e: np.where(ev == e)[0] for e in ev_list}; da = []; dd = []
    for _ in range(n):
        ma, mb = [], []
        for e in ev_list:
            g = rng.choice(groups[e], len(groups[e])); ma.append(iou_c(cA[g])); mb.append(iou_c(cB[g]))
        da.append(np.mean(ma)); dd.append(np.mean(ma) - np.mean(mb))
    ci = lambda v: (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5)))
    return float(np.mean(dd)), ci(dd)

In [ ]:

# ---------- LOEO: every recipe on every event (no checkpoint selection, final EMA weights) ----------
events = EVENTS[:2] if CFG["SMOKE"] else EVENTS
P = {r: np.zeros((N, S, S), np.float16) for r in CFG["RECIPES"]}
AUXP = {r: np.zeros(N, np.float32) for r, v in CFG["RECIPES"].items() if v["aux"]}
DONE = {r: np.zeros(N, bool) for r in CFG["RECIPES"]}
for rname, rec in CFG["RECIPES"].items():
    for e in events:
        if hours() > CFG["DEADLINE_H"]: print("time guard: stopping"); break
        held = np.where(EVENT == e)[0]; tr = np.where(EVENT != e)[0]
        pp, aa = [], []
        for s in CFG["SEEDS"]:
            m = train(rec, tr, CFG["STEPS"], s, tag=f"{rname}|{e}|s{s}")
            p, a = predict(m, held, rec["aux"]); pp.append(p.astype(np.float32)); aa.append(a); del m
        P[rname][held] = np.mean(pp, 0).astype(np.float16); DONE[rname][held] = True
        if rec["aux"]: AUXP[rname][held] = np.mean(aa, 0)
        c = counts(P[rname][held].astype(np.float32), Y[held])
        print(f"  ==> {rname} | held-out {e}: IoU {iou_c(c):.3f} precision {prec_c(c):.3f} recall {rec_c(c):.3f} | elapsed {hours():.2f} h")


In [ ]:

# ---------- results: per-event table, paired differences, post-processing ----------
ev_done = [e for e in events if all(DONE[r][EVENT == e].all() for r in CFG["RECIPES"])]
msk = np.isin(EVENT, ev_done); hi = np.where(msk)[0]
CNT = {r: counts(P[r][hi].astype(np.float32), Y[hi]) for r in CFG["RECIPES"]}
rows = []
for r in CFG["RECIPES"]:
    c = CNT[r]; per = {e: iou_c(c[EVENT[hi] == e]) for e in ev_done}; pr = {e: prec_c(c[EVENT[hi] == e]) for e in ev_done}
    rows.append(dict(recipe=r, macro_IoU=np.mean(list(per.values())), macro_precision=np.mean(list(pr.values())),
                     Somalia_prec=pr.get("Somalia", np.nan), Pakistan_prec=pr.get("Pakistan", np.nan),
                     worst_event=min(per, key=per.get), worst_IoU=min(per.values()), **{f"IoU_{e}": per[e] for e in ev_done}))
DF = pd.DataFrame(rows).set_index("recipe"); pd.set_option("display.width", 260, "display.max_columns", 40)
print("LOEO at threshold 0.5, events:", ev_done); print(DF.round(3).T.to_string())
for r in CFG["RECIPES"]:
    if r == "base": continue
    d, ci = boot_macro(CNT[r], CNT["base"], EVENT[hi], ev_done)
    print(f"paired macro-IoU {r} minus base: {d:+.4f}  95% CI [{ci[0]:+.4f}, {ci[1]:+.4f}]  (one seed: chips resampled within events, seed noise not included)")

# post-processing: remove connected components smaller than k px (k fixed grid, shown as a trend)
post = []
for r in CFG["RECIPES"]:
    for k in CFG["POST_K"]:
        pred = np.stack([drop_small(P[r][i].astype(np.float32) >= 0.5, k) for i in hi]) if k > 0 else (P[r][hi].astype(np.float32) >= 0.5)
        c = counts_bin(pred, Y[hi]); post.append(dict(recipe=r, min_component_px=k, macro_IoU=macro_iou(c, EVENT[hi], ev_done),
                                                      macro_precision=np.mean([prec_c(c[EVENT[hi] == e]) for e in ev_done]), pooled_recall=rec_c(c)))
POST = pd.DataFrame(post); print("\npost-processing (min component size):"); print(POST.pivot(index="min_component_px", columns="recipe", values="macro_IoU").round(4).to_string())
for r in AUXP:
    gate = (AUXP[r][hi] >= 0.5)
    pred = (P[r][hi].astype(np.float32) >= 0.5) & gate[:, None, None]
    c = counts_bin(pred, Y[hi]); print(f"\n{r} with chip-level water gate: macro IoU {macro_iou(c, EVENT[hi], ev_done):.4f} (ungated {DF.loc[r, 'macro_IoU']:.4f}), macro precision {np.mean([prec_c(c[EVENT[hi] == e]) for e in ev_done]):.3f}, chips gated off: {int((~gate).sum())}/{len(hi)}")
    has = np.array([(Y[i] == 1).sum() / max((Y[i] >= 0).sum(), 1) >= 0.005 for i in hi])
    print(f"  gate accuracy vs 'chip has >=0.5% water': {float(((AUXP[r][hi] >= 0.5) == has).mean()):.3f}")

DF.to_csv(f"{CFG['OUT']}/recipes_loeo_by_event.csv"); POST.to_csv(f"{CFG['OUT']}/post_processing.csv")
json.dump(dict(cfg=CFG, elapsed_h=hours()), open(f"{CFG['OUT']}/run.json", "w"), indent=1, default=str)
print("saved to", CFG["OUT"], "| total %.2f h" % hours())


## How to read it
- **Per-event precision on Somalia and Pakistan** is the target metric for "reduce false positives". A recipe that raises macro IoU only on already-good events is not what you want.
- **Post-processing table:** if macro IoU rises steadily with the minimum component size, scattered false positives are part of the problem; if it falls, the model is already clean and you are cutting real small floods.
- **Gate:** it only helps if the gate accuracy is high and the chips gated off really contain no water.
- If nothing clears the noise level, the conclusion for the paper is stronger, not weaker: single-date S1 failures on these events are not fixed by common CV tricks.